In [0]:
dbutils.widgets.text("Environment", "silver")
environment = dbutils.widgets.get("Environment")

In [0]:
from datetime import datetime
import uuid

class PipelineLogger:
    
    def __init__(self, pipeline_name):
        self.pipeline_name = pipeline_name
        self.run_id = str(uuid.uuid4())
        self.logs = []
        self.start_time = datetime.now()
        
    def log_notebook_start(self, notebook, entity):
        return {
            "pipeline_name": self.pipeline_name,
            "run_id": self.run_id,
            "notebook": notebook,
            "entity": entity,
            "start_time": datetime.now(),
            "end_time": None,
            "status": "Running",
            "error_message": None
        }
        
    def log_notebook_end(self, log_entry, status, error=None):
        log_entry["end_time"] = datetime.now()
        log_entry["status"] = status
        log_entry["error_message"] = error
        self.logs.append(log_entry)
        
    def get_logs_df(self):
        return spark.createDataFrame(self.logs)

In [0]:
def run_pipeline(metadata_df, pipeline_name):
    
    logger = PipelineLogger(pipeline_name)
    overall_status = "Succeeded"
    
    print(f"Starting pipeline: {pipeline_name}")
    print(f"Run ID: {logger.run_id}")
    
    for row in metadata_df.toLocalIterator():
        
        item = row.asDict()
        notebook_path = item["Notebook"]
        entity = item["entityName"]
        need_parameters = bool(item["NeedParameters"])
        
        log_entry = logger.log_notebook_start(notebook_path, entity)
        
        try:
            print(f"Running notebook: {notebook_path} | Entity: {entity}")
            
            if need_parameters:
                params = {
                    "ADLSaccount": item["ADLSaccount"],
                    "containerName": item["containerName"],
                    "sourceFolder": item["sourceFolder"],
                    "targetFolder": item["targetFolder"],
                    "targetEnv": item["targetEnv"],
                    "entityName": item["entityName"],
                    "fileExtension": item["fileExtension"]
                }
                dbutils.notebook.run(notebook_path, 43200, params)
            else:
                dbutils.notebook.run(notebook_path, 43200)
            
            logger.log_notebook_end(log_entry, "Succeeded")
            
        except Exception as e:
            
            overall_status = "Failed"
            logger.log_notebook_end(log_entry, "Failed", str(e))
            print(f"Failed notebook: {notebook_path}")
            break   # comportamiento igual que Synapse secuencial
    
    print(f"Pipeline finished with status: {overall_status}")
    
    logs_df = logger.get_logs_df()
    logs_df.display()
    
    if overall_status == "Failed":
        raise Exception("Pipeline execution failed")
    
    return logs_df

In [0]:
logs_df = run_pipeline(metadata_df, "01_pl_silver")